In [1]:
!pip install rdkit


In [2]:
!pip install mordred

In [3]:
from rdkit.Chem import AllChem
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

import pandas as pd
import numpy as np
from tqdm import tqdm
import os

In [4]:
dataset=pd.read_csv('/content/filtered_bioactivity_data.csv')

In [5]:
df=dataset.dropna(subset=['canonical_smiles'])

In [6]:
df

,molecule_chembl_id,canonical_smiles,bioactivity_class,MW,LogP,NumHDonors,NumHAcceptors,pIC50
0,CHEMBL1094553,[Br-].[Br-].c1ccc2c(c1)[nH]c1c[n+](Cc3ccc(C[n+...,inactive,600.358,-0.3648,2.0,0.0,4.375718
1,CHEMBL1088353,[Br-].[Br-].c1ccc2c(c1)[nH]c1c[n+](CCCC[n+]3cc...,inactive,552.314,-0.9810,2.0,0.0,4.301030
2,CHEMBL1088354,[Br-].[Br-].c1ccc2c(c1)[nH]c1c[n+](CCCCC[n+]3c...,inactive,566.341,-0.5909,2.0,0.0,4.301030
3,CHEMBL1088355,[Br-].[Br-].c1ccc2c(c1)[nH]c1c[n+](CCCCCC[n+]3...,inactive,580.368,-0.2008,2.0,0.0,4.301030
4,CHEMBL1088654,[Br-].[Br-].c1ccc2c(c1)[nH]c1c[n+](CCCCCCCC[n+...,inactive,608.422,0.5794,2.0,0.0,4.301030
...,...,...,...,...,...,...,...,...
23927,CHEMBL4749665,S=P1(c2ccccc2)OCC2CCCN21,inactive,239.280,2.1160,0.0,2.0,3.006873
23928,CHEMBL4762867,S=P1(c2ccccc2)OCCCO1,inactive,214.226,2.0583,0.0,3.0,2.893197
23929,CHEMBL4794994,S=P1(c2ccccc2)OCCO1,inactive,200.199,1.6682,0.0,3.0,3.168322
23930,CHEMBL1425,Sc1ncnc2nc[nH]c12,inactive,152.182,0.6416,2.0,4.0,3.698970


In [7]:
def RDkit_descriptors(smiles):
    mols = [Chem.MolFromSmiles(i) for i in smiles]
    calc = MoleculeDescriptors.MolecularDescriptorCalculator([x[0] for x in Descriptors._descList])
    desc_names = calc.GetDescriptorNames()

    Mol_descriptors = []
    for mol in mols:
        # add hydrogens to molecules
        mol = Chem.AddHs(mol)
        # Calculate all 200 descriptors for each molecule
        descriptors = calc.CalcDescriptors(mol)
        Mol_descriptors.append(descriptors)
    return Mol_descriptors, desc_names

# Split the SMILES into chunks of 100,000 for faster processing
chunk_size = 100000
chunks = [df[i:i+chunk_size] for i in range(0, len(df), chunk_size)]

total_chunks=len(chunks)
total_time=0

# Check if there is an existing output file
if os.path.isfile('RDkit_descriptors.csv'):
    existing_data = pd.read_csv('RDkit_descriptors.csv', index_col=0)
else:
    existing_data = pd.DataFrame()

# Calculate descriptors for each chunk and concatenate the results
for i, chunk in enumerate(tqdm(chunks, desc='Processing', total=len(chunks))):
    # Check if this chunk has already been processed
    if len(existing_data) >= len(chunk):
        continue
    # Calculate descriptors for this chunk
    descriptors, desc_names = RDkit_descriptors(chunk['canonical_smiles'])
    # Convert the descriptors to a dataframe
    df_with_200_descriptors = pd.DataFrame(descriptors, columns=desc_names,)
    # Add the chunk index as a new column
    df_with_200_descriptors['chunk_index'] = i
    # Append the data to the existing data
    existing_data = pd.concat([existing_data, df_with_200_descriptors], axis=0)
    # Save the data after each chunk
    existing_data.to_csv('RDkit_descriptors.csv')

# Save the final data
existing_data.to_csv('RDkit_descriptors.csv')

Processing: 100%|██████████| 1/1 [29:49<00:00, 1789.02s/it]
